## Extraction Games

In [1]:
import json
import re
from pathlib import Path

# Base project folder
PROJECT_ROOT = Path(r"E:\ML Projects\RAG_Chatbot_PP_MF")

CONFIG_PATH = PROJECT_ROOT / "project_files" / "data" / "json_files" / "schemes_config.json"
HTML_DIR = PROJECT_ROOT / "project_files" / "data" / "raw" / "HTML"
MD_CLEAN_DIR = PROJECT_ROOT / "project_files" / "data" / "raw" / "Markdown_cleaned"

JSON_OUTPUT_DIR = PROJECT_ROOT / "project_files" / "data" / "json_files" 
JSON_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STRUCTURED_DB_PATH = JSON_OUTPUT_DIR / "parag_parikh_structured_db.json"

print("CONFIG_PATH:        ", CONFIG_PATH)
print("HTML_DIR:           ", HTML_DIR)
print("MD_CLEAN_DIR:       ", MD_CLEAN_DIR)
print("STRUCTURED_DB_PATH: ", STRUCTURED_DB_PATH)

CONFIG_PATH:         E:\ML Projects\RAG_Chatbot_PP_MF\project_files\data\json_files\schemes_config.json
HTML_DIR:            E:\ML Projects\RAG_Chatbot_PP_MF\project_files\data\raw\HTML
MD_CLEAN_DIR:        E:\ML Projects\RAG_Chatbot_PP_MF\project_files\data\raw\Markdown_cleaned
STRUCTURED_DB_PATH:  E:\ML Projects\RAG_Chatbot_PP_MF\project_files\data\json_files\parag_parikh_structured_db.json


In [2]:
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH}")

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    schemes_config = json.load(f)

print(f"Total schemes in config: {len(schemes_config)}")
for s in schemes_config:
    print("-", s["scheme_id"])


Total schemes in config: 6
- parag-parikh-long-term-value-fund-direct-growth
- parag-parikh-liquid-fund-direct-growth
- parag-parikh-elss-tax-saver-fund-direct-growth
- parag-parikh-conservative-hybrid-fund-direct-growth
- parag-parikh-arbitrage-fund-direct-growth
- parag-parikh-dynamic-asset-allocation-fund-direct-growth


### Overview Extraction

In [3]:
def extract_overview_from_html(html: str):
    """
    Extract scheme_name, category, sub_category, risk, lock_in from the HTML.

    Based on patterns like:
      <h1 class="mfh239SchemeName displaySmall">Parag Parikh Flexi Cap Fund Direct Growth</h1>
      <div class="mfh239PillsContainer">...<span class="bodySmallHeavy">Equity</span>
      <div class="mfh239PillsContainer">...<span class="bodySmallHeavy">Flexi Cap</span>
      <div class="mfh239PillsContainer">...<span class="bodySmallHeavy">Very High Risk</span>
    """
    if not html:
        return None, None, None, None, None

    # --- Scheme name from <h1 class="mfh239SchemeName ...">...</h1> ---
    m_name = re.search(
        r'<h1[^>]*class="[^"]*mfh239SchemeName[^"]*"[^>]*>(.*?)</h1>',
        html,
        flags=re.IGNORECASE | re.DOTALL,
    )
    scheme_name = None
    if m_name:
        scheme_name = re.sub(r'\s+', ' ', m_name.group(1)).strip()

    # --- Pills: category, sub-category, risk, lock-in ---
    pill_pattern = (
        r'<div[^>]*class="[^"]*mfh239PillsContainer[^"]*"[^>]*>'
        r'.*?<span[^>]*class="[^"]*bodySmallHeavy[^"]*"[^>]*>(.*?)</span>'
    )
    pills = re.findall(pill_pattern, html, flags=re.IGNORECASE | re.DOTALL)
    pills = [re.sub(r'\s+', ' ', p).strip() for p in pills]  # normalize spaces

    # Expected order: [category, sub_category, risk, (maybe lock-in, etc.)]
    category = pills[0] if len(pills) >= 1 else None
    sub_category = pills[1] if len(pills) >= 2 else None

    # Risk: pick the pill that contains "risk" if present
    risk = next((p for p in pills if "risk" in p.lower()), None)
    if risk is None and len(pills) >= 3:
        risk = pills[2]

    # Lock-in: only if pill contains "lock in"
    lock_in = next((p for p in pills if "lock in" in p.lower()), None)

    return scheme_name, category, sub_category, risk, lock_in

In [4]:
def extract_nav_from_markdown(md: str):
    """
    NAV (date + value) from markdown.
    Regex: NAV: (.*?) (₹[\d.,]+)
    """
    if not md:
        return None

    nav_match = re.search(r'NAV:\s*(.*?)\s*(₹[\d.,]+)', md, flags=re.IGNORECASE)
    if not nav_match:
        return None

    date_part = nav_match.group(1).strip()
    value_raw = nav_match.group(2).strip()

    return {
        "raw_text": f"NAV: {date_part} {value_raw}",
        "as_on": date_part,
        "value_raw": value_raw,
    }


def extract_value_under_heading(md: str, heading: str) -> str | None:
    """
    Find a heading line and return the first non-empty line after it.
    Used for Rating, Min. SIP amount, Fund size.
    """
    if not md:
        return None

    pattern = rf'{re.escape(heading)}\s*\n+([^\n]+)'
    m = re.search(pattern, md, flags=re.IGNORECASE)
    if m:
        return m.group(1).strip()
    return None


def extract_rating(md: str) -> str | None:
    return extract_value_under_heading(md, "Rating")


def extract_min_sip(md: str) -> str | None:
    return extract_value_under_heading(md, "Min. SIP amount")


def extract_fund_size(md: str) -> str | None:
    return extract_value_under_heading(md, "Fund size")

<>:4: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\d'
C:\Users\HP\AppData\Local\Temp\ipykernel_34912\2421751044.py:4: SyntaxWarning: invalid escape sequence '\d'
  Regex: NAV: (.*?) (₹[\d.,]+)


In [5]:
structured_db = []

for scheme in schemes_config:
    scheme_id = scheme["scheme_id"]
    source_url = scheme.get("source_url")

    html_path = HTML_DIR / f"{scheme_id}.html"
    md_path = MD_CLEAN_DIR / f"{scheme_id}.md"

    # Read files (empty string if missing, to avoid crash)
    html = html_path.read_text(encoding="utf-8") if html_path.exists() else ""
    md = md_path.read_text(encoding="utf-8") if md_path.exists() else ""

    # HTML extractions
    scheme_name, category, sub_category, risk, lock_in = extract_overview_from_html(html)

    # Markdown extractions
    nav = extract_nav_from_markdown(md)
    rating = extract_rating(md)
    min_sip = extract_min_sip(md)
    fund_size = extract_fund_size(md)

    structured_db.append({
        "scheme_id": scheme_id,
        "source_url": source_url,
        "overview": {
            "scheme_name": scheme_name,
            "category": category,
            "sub_category": sub_category,
            "risk": risk,
            "lock_in": lock_in or None,
            "nav": nav,
            "rating": {"raw_value": rating} if rating is not None else None,
            "minimum_sip_amount": {"raw_value": min_sip} if min_sip is not None else None,
            "fund_size": {"raw_value": fund_size} if fund_size is not None else None,
        },
    })

print(f"Structured entries created: {len(structured_db)}")

Structured entries created: 6


In [6]:
with STRUCTURED_DB_PATH.open("w", encoding="utf-8") as f:
    json.dump(structured_db, f, ensure_ascii=False, indent=2)

print(f"✅ Saved structured DB to: {STRUCTURED_DB_PATH}")

✅ Saved structured DB to: E:\ML Projects\RAG_Chatbot_PP_MF\project_files\data\json_files\parag_parikh_structured_db.json


In [7]:
import pprint

TARGET_SCHEME_ID = "parag-parikh-long-term-value-fund-direct-growth"

for row in structured_db:
    if row["scheme_id"] == TARGET_SCHEME_ID:
        pprint.pp(row, width=120)
        break

{'scheme_id': 'parag-parikh-long-term-value-fund-direct-growth',
 'source_url': 'https://groww.in/mutual-funds/parag-parikh-long-term-value-fund-direct-growth',
 'overview': {'scheme_name': 'Parag Parikh Flexi Cap Fund Direct Growth',
              'category': 'Equity',
              'sub_category': 'Flexi Cap',
              'risk': 'Very High Risk',
              'lock_in': None,
              'nav': {'raw_text': 'NAV: 02 Jan 2026 ₹95.30', 'as_on': '02 Jan 2026', 'value_raw': '₹95.30'},
              'rating': {'raw_value': '5'},
              'minimum_sip_amount': {'raw_value': '₹1,000'},
              'fund_size': {'raw_value': '₹1,25,799.63Cr'}}}


### Holdings

In [8]:
if not STRUCTURED_DB_PATH.exists():
    raise FileNotFoundError(f"Structured DB not found: {STRUCTURED_DB_PATH}")

with STRUCTURED_DB_PATH.open("r", encoding="utf-8") as f:
    db = json.load(f)

print(f"Loaded schemes from DB: {len(db)}")

Loaded schemes from DB: 6


In [9]:
def extract_total_holdings_count(md: str) -> int | None:
    """
    Extract count from header like:
    ## Holdings (238)
    """
    if not md:
        return None

    m = re.search(r'##\s*Holdings\s*\((\d+)\)', md, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    return None


def extract_holdings_table(md: str):
    """
    Parse Markdown holdings table into structured list of dicts.
    """
    if not md:
        return []

    lines = md.splitlines()

    holdings = []
    inside_table = False

    for i, line in enumerate(lines):
        line = line.strip()

        # Detect table header
        if line.lower().startswith("name | sector | instrument | assets"):
            inside_table = True
            continue

        # Skip separator line
        if inside_table and re.match(r'^-+\|', line):
            continue

        # Stop if we hit an empty line or next section
        if inside_table and (line == "" or line.startswith("## ")):
            break

        # Parse valid row
        if inside_table and "|" in line:
            parts = [p.strip() for p in line.split("|")]
            if len(parts) == 4:
                holdings.append({
                    "name": parts[0],
                    "sector": parts[1],
                    "instrument": parts[2],
                    "assets": parts[3],
                })

    return holdings


In [10]:
for row in db:
    scheme_id = row["scheme_id"]
    md_path = MD_CLEAN_DIR / f"{scheme_id}.md"

    if not md_path.exists():
        print(f"⚠ Missing MD file for {scheme_id}")
        continue

    md = md_path.read_text(encoding="utf-8")

    # Extract holdings data
    reported_count = extract_total_holdings_count(md)
    holdings_list = extract_holdings_table(md)
    extracted_count = len(holdings_list)

    # Attach to DB
    row["holdings"] = {
        "reported_count": reported_count,
        "extracted_count": extracted_count,
        "items": holdings_list
    }

    # Sanity check output
    print(f"\nScheme: {scheme_id}")
    print(f"Reported holdings count: {reported_count}")
    print(f"Extracted holdings rows: {extracted_count}")



Scheme: parag-parikh-long-term-value-fund-direct-growth
Reported holdings count: 210
Extracted holdings rows: 210

Scheme: parag-parikh-liquid-fund-direct-growth
Reported holdings count: 47
Extracted holdings rows: 47

Scheme: parag-parikh-elss-tax-saver-fund-direct-growth
Reported holdings count: 54
Extracted holdings rows: 54

Scheme: parag-parikh-conservative-hybrid-fund-direct-growth
Reported holdings count: 139
Extracted holdings rows: 139

Scheme: parag-parikh-arbitrage-fund-direct-growth
Reported holdings count: 219
Extracted holdings rows: 219

Scheme: parag-parikh-dynamic-asset-allocation-fund-direct-growth
Reported holdings count: 200
Extracted holdings rows: 200


In [11]:
with STRUCTURED_DB_PATH.open("w", encoding="utf-8") as f:
    json.dump(db, f, ensure_ascii=False, indent=2)

print(f"\n✅ Structured DB updated with Holdings section:\n{STRUCTURED_DB_PATH}")


✅ Structured DB updated with Holdings section:
E:\ML Projects\RAG_Chatbot_PP_MF\project_files\data\json_files\parag_parikh_structured_db.json


In [12]:
if not STRUCTURED_DB_PATH.exists():
    raise FileNotFoundError(f"Structured DB not found: {STRUCTURED_DB_PATH}")

with STRUCTURED_DB_PATH.open("r", encoding="utf-8") as f:
    db = json.load(f)

print(f"Loaded schemes from DB: {len(db)}")
print("Example scheme_ids:", [row["scheme_id"] for row in db][:3])

Loaded schemes from DB: 6
Example scheme_ids: ['parag-parikh-long-term-value-fund-direct-growth', 'parag-parikh-liquid-fund-direct-growth', 'parag-parikh-elss-tax-saver-fund-direct-growth']


In [13]:
# === Cell 2: Regex-based extractor for advanced ratios from HTML ===

import re

def parse_numeric_or_null(value_str: str):
    """
    Convert a JSON-like numeric string or 'null' to Python type.
    'null' -> None
    numeric -> float or int
    """
    value_str = value_str.strip()
    if value_str == "null":
        return None
    try:
        # Try int if no decimal point
        if "." not in value_str and "e" not in value_str.lower():
            return int(value_str)
        return float(value_str)
    except ValueError:
        return None


def extract_advanced_ratios_from_html(html: str):
    """
    Extract advanced ratio fields from the HTML by regex.

    Example snippet:
      "return2y":null,"return3y":21.8,"return4y":null,"return5y":21.92,
      "return10y":18.32,"return_default":19.2,"mean_return":14.11,
      "sharpe_ratio":1.6838496478,"beta":0.56230385598241,
      "standard_deviation":8.404788082382444,"risk_rating":6,
      "risk":"Very High","return_since_created"

    We target:
      sharpe_ratio, sortino_ratio, alpha, beta,
      standard_deviation, information_ratio, risk_rating, risk
    """
    if not html:
        return None

    fields = [
        "sharpe_ratio",
        "sortino_ratio",
        "alpha",
        "beta",
        "standard_deviation",
        "information_ratio",
        "risk_rating",
    ]

    ratios = {}

    # Numeric-like fields
    for key in fields:
        pattern = rf'"{key}":(null|[-\d\.eE]+)'
        m = re.search(pattern, html)
        if m:
            ratios[key] = parse_numeric_or_null(m.group(1))
        else:
            ratios[key] = None

    # risk is a string
    m_risk = re.search(r'"risk":"([^"]+)"', html)
    ratios["risk"] = m_risk.group(1) if m_risk else None

    return ratios

In [14]:
# === Cell 3: Extract advanced ratios for each scheme & update DB ===

for row in db:
    scheme_id = row["scheme_id"]
    html_path = HTML_DIR / f"{scheme_id}.html"

    if not html_path.exists():
        print(f"⚠ Missing HTML file for {scheme_id}")
        row["advanced_ratios"] = None
        continue

    html = html_path.read_text(encoding="utf-8")

    advanced_ratios = extract_advanced_ratios_from_html(html)
    row["advanced_ratios"] = advanced_ratios

    print(f"\nScheme: {scheme_id}")
    if advanced_ratios is None:
        print("  ⚠ No advanced ratios found.")
    else:
        print("  Advanced ratios extracted:")
        for k, v in advanced_ratios.items():
            print(f"    {k}: {v}")



Scheme: parag-parikh-long-term-value-fund-direct-growth
  Advanced ratios extracted:
    sharpe_ratio: 1.8752809883
    sortino_ratio: 2.787378349468811
    alpha: 9.58759552304764
    beta: 0.55431537847881
    standard_deviation: 8.038466457808813
    information_ratio: 0.757558
    risk_rating: 6
    risk: Very High

Scheme: parag-parikh-liquid-fund-direct-growth
  Advanced ratios extracted:
    sharpe_ratio: 2.1020051508
    sortino_ratio: 3.5003060677734865
    alpha: 0.86185363342499
    beta: 0.32475732022581
    standard_deviation: 0.15015317937537837
    information_ratio: 9.0699
    risk_rating: 7
    risk: Low to Moderate

Scheme: parag-parikh-elss-tax-saver-fund-direct-growth
  Advanced ratios extracted:
    sharpe_ratio: 1.252943876
    sortino_ratio: 1.9799227998100086
    alpha: 4.90954276394985
    beta: 0.64956404160869
    standard_deviation: 9.049991766402385
    information_ratio: 0.252651
    risk_rating: 4
    risk: Moderately High

Scheme: parag-parikh-conservat

In [15]:
# === Cell 4: Save updated structured DB ===

with STRUCTURED_DB_PATH.open("w", encoding="utf-8") as f:
    json.dump(db, f, ensure_ascii=False, indent=2)

print(f"\n✅ Updated structured DB saved to:\n{STRUCTURED_DB_PATH}")


✅ Updated structured DB saved to:
E:\ML Projects\RAG_Chatbot_PP_MF\project_files\data\json_files\parag_parikh_structured_db.json


In [16]:
# === Cell 5: Print advanced_ratios for a specific scheme ===
TARGET_SCHEME_ID = "parag-parikh-long-term-value-fund-direct-growth"

for row in db:
    if row["scheme_id"] == TARGET_SCHEME_ID:
        print(f"Scheme: {row['scheme_id']}\n")
        print("Overview scheme_name:", row.get("overview", {}).get("scheme_name"))
        print("\nAdvanced Ratios:")
        pprint.pp(row.get("advanced_ratios"), width=100)
        break
else:
    print(f"Scheme not found: {TARGET_SCHEME_ID}")

Scheme: parag-parikh-long-term-value-fund-direct-growth

Overview scheme_name: Parag Parikh Flexi Cap Fund Direct Growth

Advanced Ratios:
{'sharpe_ratio': 1.8752809883,
 'sortino_ratio': 2.787378349468811,
 'alpha': 9.58759552304764,
 'beta': 0.55431537847881,
 'standard_deviation': 8.038466457808813,
 'information_ratio': 0.757558,
 'risk_rating': 6,
 'risk': 'Very High'}


### Min Investment

In [17]:
# === Cell 1: Load structured DB & set paths (Minimum Investment) ===

if not STRUCTURED_DB_PATH.exists():
    raise FileNotFoundError(f"Structured DB not found: {STRUCTURED_DB_PATH}")

with STRUCTURED_DB_PATH.open("r", encoding="utf-8") as f:
    db = json.load(f)

print(f"Loaded schemes from DB: {len(db)}")
print("Example scheme_ids:", [row["scheme_id"] for row in db][:3])


Loaded schemes from DB: 6
Example scheme_ids: ['parag-parikh-long-term-value-fund-direct-growth', 'parag-parikh-liquid-fund-direct-growth', 'parag-parikh-elss-tax-saver-fund-direct-growth']


In [18]:
# === Cell 2: Helper to extract Minimum investment amounts table from Markdown ===

import re

def extract_minimum_investment_from_markdown(md: str):
    """
    Extract minimum investment amounts from the Markdown section:

    ## Minimum investment amounts
    Min. for 1st investment | ₹1,000 
    ---|--- 
    Min. for 2nd investment onwards | ₹1,000 
    Min. for SIP | ₹1,000 

    Returns a dict:
    {
      "min_first_investment": "...",
      "min_second_investment": "...",
      "min_sip": "...",
      "rows": [
        {"label": "...", "value": "..."},
        ...
      ]
    }
    """
    if not md:
        return None

    lines = md.splitlines()

    # Step 1: find the line index of "## Minimum investment amounts"
    start_idx = None
    for i, line in enumerate(lines):
        if line.strip().lower().startswith("## minimum investment amounts"):
            start_idx = i
            break

    if start_idx is None:
        return None

    # Step 2: collect lines until blank line or next "## "
    table_lines = []
    for line in lines[start_idx + 1:]:
        stripped = line.strip()
        if stripped.startswith("## "):  # next section
            break
        if stripped == "":  # blank line
            # allow blank lines if you like; here we break
            break
        table_lines.append(stripped)

    # Step 3: parse rows "Label | Value"
    rows = []
    min_first = None
    min_second = None
    min_sip = None

    for line in table_lines:
        # Skip separator line like ---|---
        if re.match(r'^-+\s*\|\s*-+', line):
            continue
        if "|" not in line:
            continue

        parts = [p.strip() for p in line.split("|")]
        if len(parts) < 2:
            continue

        label = parts[0]
        value = parts[1]

        rows.append({"label": label, "value": value})

        label_lower = label.lower()

        if "1st investment" in label_lower or "first investment" in label_lower:
            min_first = value
        elif "2nd investment onwards" in label_lower or "second investment onwards" in label_lower:
            min_second = value
        elif "min. for sip" in label_lower or "min for sip" in label_lower or "sip" in label_lower:
            min_sip = value

    if not rows:
        return None

    return {
        "min_first_investment": min_first,
        "min_second_investment": min_second,
        "min_sip": min_sip,
        "rows": rows,
    }


In [19]:
# === Cell 3: Extract Minimum Investment for each scheme & update DB ===

for row in db:
    scheme_id = row["scheme_id"]
    md_path = MD_CLEAN_DIR / f"{scheme_id}.md"

    if not md_path.exists():
        print(f"⚠ Missing MD file for {scheme_id}")
        row["minimum_investment"] = None
        continue

    md = md_path.read_text(encoding="utf-8")

    min_inv = extract_minimum_investment_from_markdown(md)
    row["minimum_investment"] = min_inv

    print(f"\nScheme: {scheme_id}")
    if min_inv is None:
        print("  ⚠ No 'Minimum investment amounts' section found.")
    else:
        print("  Min. for 1st investment:        ", min_inv.get("min_first_investment"))
        print("  Min. for 2nd investment onwards:", min_inv.get("min_second_investment"))
        print("  Min. for SIP:                   ", min_inv.get("min_sip"))



Scheme: parag-parikh-long-term-value-fund-direct-growth
  Min. for 1st investment:         ₹1,000
  Min. for 2nd investment onwards: ₹1,000
  Min. for SIP:                    ₹1,000

Scheme: parag-parikh-liquid-fund-direct-growth
  Min. for 1st investment:         ₹5,000
  Min. for 2nd investment onwards: ₹1,000
  Min. for SIP:                    ₹1,000

Scheme: parag-parikh-elss-tax-saver-fund-direct-growth
  Min. for 1st investment:         ₹500
  Min. for 2nd investment onwards: ₹500
  Min. for SIP:                    ₹1,000

Scheme: parag-parikh-conservative-hybrid-fund-direct-growth
  Min. for 1st investment:         ₹5,000
  Min. for 2nd investment onwards: ₹1,000
  Min. for SIP:                    ₹1,000

Scheme: parag-parikh-arbitrage-fund-direct-growth
  Min. for 1st investment:         ₹1,000
  Min. for 2nd investment onwards: ₹1,000
  Min. for SIP:                    ₹1,000

Scheme: parag-parikh-dynamic-asset-allocation-fund-direct-growth
  Min. for 1st investment:         

In [20]:
# === Cell 4: Save updated structured DB with minimum_investment ===

with STRUCTURED_DB_PATH.open("w", encoding="utf-8") as f:
    json.dump(db, f, ensure_ascii=False, indent=2)

print(f"\n✅ Updated structured DB saved to:\n{STRUCTURED_DB_PATH}")



✅ Updated structured DB saved to:
E:\ML Projects\RAG_Chatbot_PP_MF\project_files\data\json_files\parag_parikh_structured_db.json


In [21]:
# === Cell 5: Print Minimum Investment section for a specific scheme ===

TARGET_SCHEME_ID = "parag-parikh-long-term-value-fund-direct-growth"

for row in db:
    if row["scheme_id"] == TARGET_SCHEME_ID:
        print(f"Scheme: {row['scheme_id']}\n")
        print("Overview scheme_name:", row.get("overview", {}).get("scheme_name"))
        print("\nMinimum Investment:")
        pprint.pp(row.get("minimum_investment"), width=100)
        break
else:
    print(f"Scheme not found: {TARGET_SCHEME_ID}")

Scheme: parag-parikh-long-term-value-fund-direct-growth

Overview scheme_name: Parag Parikh Flexi Cap Fund Direct Growth

Minimum Investment:
{'min_first_investment': '₹1,000',
 'min_second_investment': '₹1,000',
 'min_sip': '₹1,000',
 'rows': [{'label': 'Min. for 1st investment', 'value': '₹1,000'},
          {'label': 'Min. for 2nd investment onwards', 'value': '₹1,000'},
          {'label': 'Min. for SIP', 'value': '₹1,000'}]}


### Returns and Rankings

In [22]:
# === Cell 1: Load structured DB & set paths (Returns & Rankings) ===
if not STRUCTURED_DB_PATH.exists():
    raise FileNotFoundError(f"Structured DB not found: {STRUCTURED_DB_PATH}")

with STRUCTURED_DB_PATH.open("r", encoding="utf-8") as f:
    db = json.load(f)

print(f"Loaded schemes from DB: {len(db)}")
print("Example scheme_ids:", [row["scheme_id"] for row in db][:3])


Loaded schemes from DB: 6
Example scheme_ids: ['parag-parikh-long-term-value-fund-direct-growth', 'parag-parikh-liquid-fund-direct-growth', 'parag-parikh-elss-tax-saver-fund-direct-growth']


In [23]:
# === Cell 2: Full Returns & Rankings extractor (UI style) ===

def extract_returns_and_rankings_from_markdown(md: str):
    """
    Extract the entire Returns & Rankings table as shown in UI.
    """
    if not md:
        return None

    lines = md.splitlines()

    # 1. Locate section
    start_idx = None
    for i, line in enumerate(lines):
        if line.strip().lower().startswith("## returns and rankings"):
            start_idx = i
            break

    if start_idx is None:
        return None

    category = None
    available_types = []
    active_type = None
    header_line = None
    header_idx = None

    # 2. Scan for: category, types, header
    for i in range(start_idx + 1, len(lines)):
        line = lines[i].strip()

        # Stop at next section
        if line.startswith("## "):
            break

        # Find category
        if line.lower().startswith("category:"):
            category = line.split(":",1)[1].strip()
            continue

        # Annualised / Absolute return types
        if line.lower() in ["annualised returns", "absolute returns"]:
            available_types.append(line)
            if active_type is None:
                active_type = line
            continue

        # find markdown table header
        if line.startswith("|") and "|" in line[1:]:
            header_line = line
            header_idx = i
            break

    if header_line is None:
        return {
            "category": category,
            "available_types": available_types,
            "active_type": active_type,
            "table": None
        }

    # 3. Parse columns
    columns = [c.strip() for c in header_line.strip("|").split("|")]

    # 4. Parse rows
    rows = []
    i = header_idx + 1

    # skip separator
    if i < len(lines) and re.match(r'^-+\|', lines[i]):
        i += 1

    while i < len(lines):
        line = lines[i].strip()
        if line == "" or line.startswith("## "):
            break
        if "|" not in line:
            i += 1
            continue
        
        parts = [p.strip() for p in line.strip("|").split("|")]
        label = parts[0]
        values = parts[1:]

        rows.append({
            "label": label,
            "values": values
        })

        i += 1

    return {
        "category": category,
        "available_types": available_types,
        "active_type": active_type,
        "table": {
            "columns": columns,
            "rows": rows
        }
    }

In [24]:
# === Cell 3: Extract full Returns & Rankings for each scheme ===

for row in db:
    scheme_id = row["scheme_id"]
    md_path = MD_CLEAN_DIR / f"{scheme_id}.md"

    if not md_path.exists():
        print(f"⚠ No MD for {scheme_id}")
        row["returns_and_rankings"] = None
        continue

    md = md_path.read_text(encoding="utf-8")

    rr = extract_returns_and_rankings_from_markdown(md)
    row["returns_and_rankings"] = rr

    print(f"\nScheme: {scheme_id}")
    print("  Category:", rr.get("category"))
    print("  Active type:", rr.get("active_type"))
    print("  Columns:", rr.get("table",{}).get("columns"))


Scheme: parag-parikh-long-term-value-fund-direct-growth
  Category: Equity Flexi Cap
  Active type: Annualised returns
  Columns: ['3Y', '5Y', '10Y', 'All']

Scheme: parag-parikh-liquid-fund-direct-growth
  Category: Debt Liquid
  Active type: Annualised returns
  Columns: ['1Y', '3Y', '5Y', 'All']

Scheme: parag-parikh-elss-tax-saver-fund-direct-growth
  Category: Equity ELSS
  Active type: Annualised returns
  Columns: ['1Y', '3Y', '5Y', 'All']

Scheme: parag-parikh-conservative-hybrid-fund-direct-growth
  Category: Hybrid Conservative Hybrid
  Active type: Annualised returns
  Columns: ['6M', '1Y', '3Y', 'All']

Scheme: parag-parikh-arbitrage-fund-direct-growth
  Category: Hybrid Arbitrage
  Active type: Absolute returns
  Columns: ['3M', '6M', '1Y', 'All']

Scheme: parag-parikh-dynamic-asset-allocation-fund-direct-growth
  Category: Hybrid Dynamic Asset Allocation
  Active type: Absolute returns
  Columns: ['3M', '6M', '1Y', 'All']


In [25]:
# === Cell 4: Save updated DB ===

with STRUCTURED_DB_PATH.open("w", encoding="utf-8") as f:
    json.dump(db, f, ensure_ascii=False, indent=2)

print("✅ Saved")

✅ Saved


In [26]:
# === Cell 5: Print Returns & Rankings section for a specific scheme ===

TARGET_SCHEME_ID = "parag-parikh-long-term-value-fund-direct-growth"

for row in db:
    if row["scheme_id"] == TARGET_SCHEME_ID:
        print(f"Scheme: {row['scheme_id']}\n")
        print("Overview scheme_name:", row.get("overview", {}).get("scheme_name"))
        print("\nReturns & Rankings:")
        pprint.pp(row.get("returns_and_rankings"), width=120)
        break
else:
    print(f"Scheme not found: {TARGET_SCHEME_ID}")

Scheme: parag-parikh-long-term-value-fund-direct-growth

Overview scheme_name: Parag Parikh Flexi Cap Fund Direct Growth

Returns & Rankings:
{'category': 'Equity Flexi Cap',
 'available_types': ['Annualised returns', 'Absolute returns'],
 'active_type': 'Annualised returns',
 'table': {'columns': ['3Y', '5Y', '10Y', 'All'],
           'rows': [{'label': 'Fund returns', 'values': ['23.1%', '20.7%', '18.3%', '19.6%']},
                    {'label': 'Category average', 'values': ['16.0%', '15.6%', '13.5%', 'NA']},
                    {'label': 'Rank with in category', 'values': ['7', '10', '1', 'NA']}]}}


### Expense ratio, exit load and tax

In [27]:
# === Cell 1: Load structured DB & set paths (Expense ratio, exit load and tax) ===

if not STRUCTURED_DB_PATH.exists():
    raise FileNotFoundError(f"Structured DB not found: {STRUCTURED_DB_PATH}")

with STRUCTURED_DB_PATH.open("r", encoding="utf-8") as f:
    db = json.load(f)

print(f"Loaded schemes from DB: {len(db)}")
print("Example scheme_ids:", [row["scheme_id"] for row in db][:3])


Loaded schemes from DB: 6
Example scheme_ids: ['parag-parikh-long-term-value-fund-direct-growth', 'parag-parikh-liquid-fund-direct-growth', 'parag-parikh-elss-tax-saver-fund-direct-growth']


In [28]:
# === Cell 2: Helpers to parse 'Expense ratio, exit load and tax' section ===

def extract_expense_ratio(md: str):
    """
    Get the value after '### Expense ratio:' from the markdown.
    Example line:
      '### Expense ratio: 0.30%'
    Returns: '0.30%' (string) or None
    """
    if not md:
        return None

    pattern = r'###\s*Expense ratio:\s*([^\n]+)'
    m = re.search(pattern, md, flags=re.IGNORECASE)
    if m:
        return m.group(1).strip()
    return None


def extract_block_under_heading(md: str, heading: str):
    """
    Get all text lines under a '### <heading>' until next '### ' or '## '.

    Example:
      ### Exit load
      Exit load of 0.25%, if redeemed within 30 days.
      <stop when next heading or blank line / EOF>

    Returns a single string with lines joined by spaces, or None.
    """
    if not md:
        return None

    lines = md.splitlines()
    start_idx = None

    # Find '### <heading>' (case-insensitive, no colon here)
    heading_lower = heading.lower()
    for i, line in enumerate(lines):
        stripped = line.strip()
        if stripped.lower().startswith(f"### {heading_lower}"):
            start_idx = i
            break

    if start_idx is None:
        return None

    collected = []
    for line in lines[start_idx + 1:]:
        stripped = line.strip()

        # Stop at next heading
        if stripped.startswith("### ") or stripped.startswith("## "):
            break

        # Skip purely empty lines if you want, or keep; here we skip empties
        if stripped == "":
            continue

        collected.append(stripped)

    if not collected:
        return None

    # Join lines with a space so it's one clean text block
    return " ".join(collected)


In [29]:
# === Cell 3: Extract Expense ratio, Exit load, Stamp duty, Tax implication & update DB ===

for row in db:
    scheme_id = row["scheme_id"]
    md_path = MD_CLEAN_DIR / f"{scheme_id}.md"

    if not md_path.exists():
        print(f"⚠ Missing MD file for {scheme_id}")
        row["expense_exit_tax"] = None
        continue

    md = md_path.read_text(encoding="utf-8")

    expense_ratio = extract_expense_ratio(md)
    exit_load = extract_block_under_heading(md, "Exit load")
    stamp_duty = extract_block_under_heading(md, "Stamp duty")
    tax_implication = extract_block_under_heading(md, "Tax implication")

    row["expense_exit_tax"] = {
        "expense_ratio": expense_ratio,
        "exit_load": exit_load,
        "stamp_duty": stamp_duty,
        "tax_implication": tax_implication,
    }

    print(f"\nScheme: {scheme_id}")
    print("  Expense ratio: ", expense_ratio)
    print("  Exit load:     ", exit_load)
    print("  Stamp duty:    ", stamp_duty)
    print("  Tax implication:", tax_implication[:80] + "..." if tax_implication and len(tax_implication) > 80 else tax_implication)



Scheme: parag-parikh-long-term-value-fund-direct-growth
  Expense ratio:  0.63%
  Exit load:      For units above 10% of the investment, exit load of 2% if redeemed within 365 days and 1% if redeemed after 365 days but on or before 730 days.
  Stamp duty:     0.005% (from July 1st, 2020)
  Tax implication: Returns are taxed at 20%, if you redeem before one year. After 1 year, you are r...

Scheme: parag-parikh-liquid-fund-direct-growth
  Expense ratio:  0.11%
  Exit load:      Exit load of 0.0070% if redeemed within 1 day, 0.0065% if redeemed within 2 days, 0.0060% if redeemed within 3 days, 0.0055% if redeemed within 4 days, 0.0050% if redeemed within 5 days, 0.0045% if redeemed within 6 days.
  Stamp duty:     0.005% (from July 1st, 2020)
  Tax implication: Returns are taxed as per your Income Tax slab. Understand terms Check past data

Scheme: parag-parikh-elss-tax-saver-fund-direct-growth
  Expense ratio:  0.62%
  Exit load:      Nil
  Stamp duty:     0.005% (from July 1st, 2020)


In [30]:
# === Cell 4: Save updated structured DB with expense_exit_tax ===

with STRUCTURED_DB_PATH.open("w", encoding="utf-8") as f:
    json.dump(db, f, ensure_ascii=False, indent=2)

print(f"\n✅ Updated structured DB saved to:\n{STRUCTURED_DB_PATH}")



✅ Updated structured DB saved to:
E:\ML Projects\RAG_Chatbot_PP_MF\project_files\data\json_files\parag_parikh_structured_db.json


In [31]:
# === Cell 5: Print Expense/Exit/Tax section for a specific scheme ===

TARGET_SCHEME_ID = "parag-parikh-long-term-value-fund-direct-growth"

for row in db:
    if row["scheme_id"] == TARGET_SCHEME_ID:
        print(f"Scheme: {row['scheme_id']}\n")
        print("Overview scheme_name:", row.get("overview", {}).get("scheme_name"))
        print("\nExpense ratio, exit load and tax:")
        pprint.pp(row.get("expense_exit_tax"), width=120)
        break
else:
    print(f"Scheme not found: {TARGET_SCHEME_ID}")


Scheme: parag-parikh-long-term-value-fund-direct-growth

Overview scheme_name: Parag Parikh Flexi Cap Fund Direct Growth

Expense ratio, exit load and tax:
{'expense_ratio': '0.63%',
 'exit_load': 'For units above 10% of the investment, exit load of 2% if redeemed within 365 days and 1% if redeemed '
              'after 365 days but on or before 730 days.',
 'stamp_duty': '0.005% (from July 1st, 2020)',
 'tax_implication': 'Returns are taxed at 20%, if you redeem before one year. After 1 year, you are required to pay '
                    'LTCG tax of 12.5% on returns of Rs 1.25 lakh+ in a financial year. Understand terms Check past '
                    'data'}


### Fund Management 

In [32]:
# === Cell 1: Load structured DB & set paths (Fund Management) ===

if not STRUCTURED_DB_PATH.exists():
    raise FileNotFoundError(f"Structured DB not found: {STRUCTURED_DB_PATH}")

with STRUCTURED_DB_PATH.open("r", encoding="utf-8") as f:
    db = json.load(f)

print(f"Loaded schemes from DB: {len(db)}")
print("Example scheme_ids:", [row["scheme_id"] for row in db][:3])


Loaded schemes from DB: 6
Example scheme_ids: ['parag-parikh-long-term-value-fund-direct-growth', 'parag-parikh-liquid-fund-direct-growth', 'parag-parikh-elss-tax-saver-fund-direct-growth']


In [33]:
# === Cell 2: Helpers to parse 'Fund management' section ===

def parse_manager_block(block_lines):
    """
    Parse a single manager block, which looks like:

    ### 
    Raj Mehta
    Oct 2023 - Present
    View details
    Education
    ... education text ...
    Experience
    ... experience text ...
    Also manages these schemes
    ...

    or possibly '### Raj Mehta' with name inline.

    Returns a dict {name, tenure, education, experience} or None.
    """
    if not block_lines:
        return None

    name = None
    tenure = None
    education_lines = []
    experience_lines = []

    # ---- Pass 1: name + tenure ----
    for idx, raw in enumerate(block_lines):
        s = raw.strip()
        if not s:
            continue

        # First line may be '### ...'
        if idx == 0 and s.startswith("###"):
            inline = s[3:].strip(" #")
            if inline:
                name = inline
            continue

        low = s.lower()

        # Skip markers when searching for name/tenure
        if low in ["view details"] or low.startswith("education") or low.startswith("experience") or low.startswith("also manages"):
            continue

        if not name:
            name = s
            continue

        if name and not tenure:
            tenure = s
            continue

    # ---- Pass 2: education + experience sections ----
    mode = None  # 'edu' or 'exp'
    for idx, raw in enumerate(block_lines):
        s = raw.strip()
        low = s.lower()

        if idx == 0 and s.startswith("###"):
            continue

        if low.startswith("education"):
            mode = "edu"
            continue
        if low.startswith("experience"):
            mode = "exp"
            continue
        if low.startswith("also manages"):
            mode = None
            continue
        if s.startswith("###") or s.startswith("## "):
            mode = None
            continue

        if mode == "edu":
            if s:
                education_lines.append(s)
        elif mode == "exp":
            if s:
                experience_lines.append(s)

    education = " ".join(education_lines) if education_lines else None
    experience = " ".join(experience_lines) if experience_lines else None

    # If we have nothing meaningful, skip
    if not (name or tenure or education or experience):
        return None

    return {
        "name": name,
        "tenure": tenure,
        "education": education,
        "experience": experience,
    }


def extract_fund_management_from_markdown(md: str):
    """
    Extract the full 'Fund management' section into a list of managers.

    Structure:

    ## Fund management
    ### 
    <manager 1 block>
    ### 
    <manager 2 block>
    ...
    ### 
    ## Fund house & investment objective   (next section)

    Returns:
    {
      "managers": [ {name, tenure, education, experience}, ... ],
      "manager_names": "Name1, Name2, ..."
    }
    or None if no section found.
    """
    if not md:
        return None

    lines = md.splitlines()

    # Find '## Fund management'
    start_idx = None
    for i, line in enumerate(lines):
        if line.strip().lower().startswith("## fund management"):
            start_idx = i
            break

    if start_idx is None:
        return None

    # Find end index: next '## ' after start
    end_idx = len(lines)
    for i in range(start_idx + 1, len(lines)):
        if lines[i].strip().startswith("## "):
            end_idx = i
            break

    block = lines[start_idx + 1:end_idx]

    # Find indices where manager blocks start (lines starting with '###')
    manager_start_indices = []
    for idx, line in enumerate(block):
        if line.strip().startswith("###"):
            manager_start_indices.append(idx)

    if not manager_start_indices:
        return None

    manager_start_indices.append(len(block))  # sentinel for last block

    managers = []

    for pos in range(len(manager_start_indices) - 1):
        start = manager_start_indices[pos]
        end = manager_start_indices[pos + 1]
        manager_block_lines = block[start:end]

        manager = parse_manager_block(manager_block_lines)
        if manager is not None:
            managers.append(manager)

    if not managers:
        return None

    manager_names = ", ".join(
        m["name"] for m in managers if m.get("name")
    ) or None

    return {
        "managers": managers,
        "manager_names": manager_names,
    }


In [34]:
# === Cell 3: Extract Fund Management for each scheme & update DB ===

for row in db:
    scheme_id = row["scheme_id"]
    md_path = MD_CLEAN_DIR / f"{scheme_id}.md"

    if not md_path.exists():
        print(f"⚠ Missing MD file for {scheme_id}")
        row["fund_management"] = None
        continue

    md = md_path.read_text(encoding="utf-8")

    fm = extract_fund_management_from_markdown(md)
    row["fund_management"] = fm

    print(f"\nScheme: {scheme_id}")
    if fm is None:
        print("  ⚠ No 'Fund management' section found.")
    else:
        print("  Manager names:", fm.get("manager_names"))
        print("  Total managers parsed:", len(fm.get("managers", [])))
        # print brief of first manager
        if fm["managers"]:
            first = fm["managers"][0]
            print("   First manager:", first.get("name"), "| Tenure:", first.get("tenure"))



Scheme: parag-parikh-long-term-value-fund-direct-growth
  Manager names: Rajeev Thakkar, Raunak Onkar, Raj Mehta, Rukun Tarachandani, Tejas Soman, Mansi Kariya, Aishwarya Dhar
  Total managers parsed: 7
   First manager: Rajeev Thakkar | Tenure: May 2013 - Present

Scheme: parag-parikh-liquid-fund-direct-growth
  Manager names: Tejas Soman, Mansi Kariya, Aishwarya Dhar
  Total managers parsed: 3
   First manager: Tejas Soman | Tenure: Sep 2025 - Present

Scheme: parag-parikh-elss-tax-saver-fund-direct-growth
  Manager names: Rajeev Thakkar, Raunak Onkar, Raj Mehta, Rukun Tarachandani, Tejas Soman, Aishwarya Dhar
  Total managers parsed: 6
   First manager: Rajeev Thakkar | Tenure: Jul 2019 - Present

Scheme: parag-parikh-conservative-hybrid-fund-direct-growth
  Manager names: Rajeev Thakkar, Raunak Onkar, Raj Mehta, Rukun Tarachandani, Tejas Soman, Mansi Kariya
  Total managers parsed: 6
   First manager: Rajeev Thakkar | Tenure: May 2021 - Present

Scheme: parag-parikh-arbitrage-fund

In [35]:
# === Cell 4: Save updated structured DB with fund_management ===

with STRUCTURED_DB_PATH.open("w", encoding="utf-8") as f:
    json.dump(db, f, ensure_ascii=False, indent=2)

print(f"\n✅ Updated structured DB saved to:\n{STRUCTURED_DB_PATH}")


✅ Updated structured DB saved to:
E:\ML Projects\RAG_Chatbot_PP_MF\project_files\data\json_files\parag_parikh_structured_db.json


In [36]:
# === Cell 5: Print Fund Management section for a specific scheme ===

TARGET_SCHEME_ID = "parag-parikh-long-term-value-fund-direct-growth"

for row in db:
    if row["scheme_id"] == TARGET_SCHEME_ID:
        print(f"Scheme: {row['scheme_id']}\n")
        print("Overview scheme_name:", row.get("overview", {}).get("scheme_name"))
        print("\nFund Management:")
        pprint.pp(row.get("fund_management"), width=120)
        break
else:
    print(f"Scheme not found: {TARGET_SCHEME_ID}")


Scheme: parag-parikh-long-term-value-fund-direct-growth

Overview scheme_name: Parag Parikh Flexi Cap Fund Direct Growth

Fund Management:
{'managers': [{'name': 'Rajeev Thakkar',
               'tenure': 'May 2013 - Present',
               'education': 'Mr.Thakkar is a Chartered Accountant, Cost Accountant, CFA, and CFP.',
               'experience': 'He has been associated with PPFAS AMC since 2013.'},
              {'name': 'Raunak Onkar',
               'tenure': 'May 2013 - Present',
               'education': 'Mr.Onkar is an MMS (Finance) from the University of Mumbai.',
               'experience': 'He joined PPFAS as a research analyst. He was appointed as Head- research in the year '
                             '2011. He is working with the company as an Associate Fund Manager.'},
              {'name': 'Raj Mehta',
               'tenure': 'Jan 2016 - Present',
               'education': 'Mr. Mehta is a Commerce Graduate from University of Mumbai (N.M. College) ; Fellow 

### Fund house & investment objective / contact

In [37]:
# === Cell 1: Load structured DB & set paths (Fund house & investment objective / contact) ===

if not STRUCTURED_DB_PATH.exists():
    raise FileNotFoundError(f"Structured DB not found: {STRUCTURED_DB_PATH}")

with STRUCTURED_DB_PATH.open("r", encoding="utf-8") as f:
    db = json.load(f)

print(f"Loaded schemes from DB: {len(db)}")
print("Example scheme_ids:", [row["scheme_id"] for row in db][:3])


Loaded schemes from DB: 6
Example scheme_ids: ['parag-parikh-long-term-value-fund-direct-growth', 'parag-parikh-liquid-fund-direct-growth', 'parag-parikh-elss-tax-saver-fund-direct-growth']


In [38]:
# === Cell 2: Helper to extract a section block between '##' headings ===

def extract_section_block(md: str, heading: str):
    """
    Extract text under a '## <heading>' until the next line that starts with '## '.

    Example:
      ## Fund house & investment objective
      ... lines ...
      ## Fund house contact details

    call: extract_section_block(md, "Fund house & investment objective")

    Returns a single string (with newlines) or None.
    """
    if not md:
        return None

    lines = md.splitlines()
    heading_lower = heading.lower()

    start_idx = None
    for i, line in enumerate(lines):
        if line.strip().lower().startswith("## " + heading_lower):
            start_idx = i
            break

    if start_idx is None:
        return None

    # Collect until next '## '
    collected = []
    for line in lines[start_idx + 1:]:
        stripped = line.rstrip("\n")
        if stripped.strip().startswith("## "):  # next section starts
            break
        collected.append(stripped)

    # Strip leading/trailing empty lines
    while collected and collected[0].strip() == "":
        collected.pop(0)
    while collected and collected[-1].strip() == "":
        collected.pop()

    if not collected:
        return None

    return "\n".join(collected)


In [39]:
# === Cell 3: Extract Fund house & investment objective / contact details & update DB ===

for row in db:
    scheme_id = row["scheme_id"]
    md_path = MD_CLEAN_DIR / f"{scheme_id}.md"

    if not md_path.exists():
        print(f"⚠ Missing MD file for {scheme_id}")
        row["fund_house_and_investment_objective"] = None
        row["fund_house_contact_details"] = None
        continue

    md = md_path.read_text(encoding="utf-8")

    fh_obj = extract_section_block(md, "Fund house & investment objective")
    fh_contact = extract_section_block(md, "Fund house contact details")

    row["fund_house_and_investment_objective"] = fh_obj
    row["fund_house_contact_details"] = fh_contact

    print(f"\nScheme: {scheme_id}")
    print("  Fund house & investment objective: ",
          (fh_obj[:80] + "...") if fh_obj and len(fh_obj) > 80 else fh_obj)
    print("  Fund house contact details:        ",
          (fh_contact[:80] + "...") if fh_contact and len(fh_contact) > 80 else fh_contact)



Scheme: parag-parikh-long-term-value-fund-direct-growth
  Fund house & investment objective:   Amc Image PPFAS Mutual Fund 
Rank (total assets) | #23 in India 
---|--- 
Total...
  Fund house contact details:         Address
702-705, 7th Floor, 349 Business Point Commercial Premises Co-op. Societ...

Scheme: parag-parikh-liquid-fund-direct-growth
  Fund house & investment objective:   Amc Image PPFAS Mutual Fund 
Rank (total assets) | #23 in India 
---|--- 
Total...
  Fund house contact details:         Address
702-705, 7th Floor, 349 Business Point Commercial Premises Co-op. Societ...

Scheme: parag-parikh-elss-tax-saver-fund-direct-growth
  Fund house & investment objective:   Amc Image PPFAS Mutual Fund 
Rank (total assets) | #23 in India 
---|--- 
Total...
  Fund house contact details:         Address
702-705, 7th Floor, 349 Business Point Commercial Premises Co-op. Societ...

Scheme: parag-parikh-conservative-hybrid-fund-direct-growth
  Fund house & investment objective:   Amc Ima

In [40]:
# === Cell 4: Save updated structured DB with fund_house sections ===

with STRUCTURED_DB_PATH.open("w", encoding="utf-8") as f:
    json.dump(db, f, ensure_ascii=False, indent=2)

print(f"\n✅ Updated structured DB saved to:\n{STRUCTURED_DB_PATH}")



✅ Updated structured DB saved to:
E:\ML Projects\RAG_Chatbot_PP_MF\project_files\data\json_files\parag_parikh_structured_db.json


In [41]:
# === Cell 5: Print Fund house sections for a specific scheme ===

TARGET_SCHEME_ID = "parag-parikh-long-term-value-fund-direct-growth"

for row in db:
    if row["scheme_id"] == TARGET_SCHEME_ID:
        print(f"Scheme: {row['scheme_id']}\n")
        print("Overview scheme_name:", row.get("overview", {}).get("scheme_name"))

        print("\nFund house & investment objective:\n")
        print(row.get("fund_house_and_investment_objective"))

        print("\nFund house contact details:\n")
        print(row.get("fund_house_contact_details"))
        break
else:
    print(f"Scheme not found: {TARGET_SCHEME_ID}")

Scheme: parag-parikh-long-term-value-fund-direct-growth

Overview scheme_name: Parag Parikh Flexi Cap Fund Direct Growth

Fund house & investment objective:

 Amc Image PPFAS Mutual Fund 
Rank (total assets) | #23 in India 
---|--- 
Total AUM | ₹1,48,568.37Cr 
Date of Incorporation | 10 Oct 2012 
---|--- 
Parag Parikh Flexi Cap Fund Direct Growth is a Equity Mutual Fund Scheme launched by PPFAS Mutual Fund. This scheme was made available to inve
Parag Parikh Flexi Cap Fund Direct Growth is a Equity Mutual Fund Scheme launched by PPFAS Mutual Fund. This scheme was made available to investors on 10 Oct 2012. Raunak Onkar, Rajeev Thakkar, Raj Mehta is the Current Fund Manager of Parag Parikh Flexi Cap Fund Direct Growth fund. The fund currently has an Asset Under Management(AUM) of ₹1,48,568 Cr and the Latest NAV as of 02 Jan 2026 is ₹95.30. The Parag Parikh Flexi Cap Fund Direct Growth is rated Very High risk. Minimum SIP Investment is set to ₹1,000. Minimum Lumpsum Investment is ₹1,000.

### FAQs

In [42]:
# === Cell 1: Load structured DB & set paths (FAQs) ===

if not STRUCTURED_DB_PATH.exists():
    raise FileNotFoundError(f"Structured DB not found: {STRUCTURED_DB_PATH}")

with STRUCTURED_DB_PATH.open("r", encoding="utf-8") as f:
    db = json.load(f)

print(f"Loaded schemes from DB: {len(db)}")
print("Example scheme_ids:", [row["scheme_id"] for row in db][:3])


Loaded schemes from DB: 6
Example scheme_ids: ['parag-parikh-long-term-value-fund-direct-growth', 'parag-parikh-liquid-fund-direct-growth', 'parag-parikh-elss-tax-saver-fund-direct-growth']


In [43]:
# === Cell 2: Helper to extract FAQs from Markdown (with last FAQ truncation) ===

def extract_faqs_from_markdown(md: str):
    """
    Extract FAQ Q&A pairs from markdown section:

    ## FAQs
    ### Question
    Answer ...

    NEW RULE:
    - Last FAQ answer may run until EOF (no next ##)
    - Therefore, TRIM last FAQ answer to 50–60 words (clean)
    """

    if not md:
        return None

    lines = md.splitlines()

    # 1) Find '## FAQs'
    start_idx = None
    for i, line in enumerate(lines):
        if line.strip().lower().startswith("## faqs"):
            start_idx = i
            break

    if start_idx is None:
        return None

    # 2) Find end (next ##)
    end_idx = len(lines)
    for i in range(start_idx + 1, len(lines)):
        if lines[i].strip().startswith("## "):
            end_idx = i
            break

    faq_block = lines[start_idx + 1:end_idx]

    # 3) Split into Q&A
    items = []
    current_question = None
    current_answer_lines = []

    def flush_current(is_last=False):
        """Save one Q&A. If last: trim to 50–60 words."""
        if current_question is None:
            return

        # Clean empty edges
        while current_answer_lines and current_answer_lines[0].strip() == "":
            current_answer_lines.pop(0)
        while current_answer_lines and current_answer_lines[-1].strip() == "":
            current_answer_lines.pop()

        answer_text = "\n".join(current_answer_lines)

        if is_last:
            # Trim to around 50–60 words
            words = answer_text.split()
            if len(words) > 60:
                answer_text = " ".join(words[:60]) + "..."

        items.append({
            "question": current_question,
            "answer": answer_text,
        })

    # Parse lines
    for line in faq_block:
        stripped = line.strip()
        if stripped.startswith("### "):
            # New question -> flush previous
            if current_question is not None:
                flush_current(is_last=False)
            current_question = stripped[4:].strip()
            current_answer_lines = []
        else:
            if current_question is not None:
                current_answer_lines.append(line)

    # Flush *last* FAQ (apply trimming!)
    flush_current(is_last=True)

    if not items:
        return None

    return {
        "items": items,
        "count": len(items),
    }

In [44]:
# === Cell 3: Extract FAQs for each scheme & update DB ===

for row in db:
    scheme_id = row["scheme_id"]
    md_path = MD_CLEAN_DIR / f"{scheme_id}.md"

    if not md_path.exists():
        print(f"⚠ Missing MD file for {scheme_id}")
        row["faqs"] = None
        continue

    md = md_path.read_text(encoding="utf-8")

    faqs = extract_faqs_from_markdown(md)
    row["faqs"] = faqs

    print(f"\nScheme: {scheme_id}")
    if faqs is None:
        print("  ⚠ No 'FAQs' section found.")
    else:
        print("  FAQ count:", faqs.get("count"))
        if faqs["items"]:
            print("   First question:", faqs["items"][0]["question"])



Scheme: parag-parikh-long-term-value-fund-direct-growth
  FAQ count: 8
   First question: How to Invest in Parag Parikh Flexi Cap Fund Direct Growth?

Scheme: parag-parikh-liquid-fund-direct-growth
  FAQ count: 8
   First question: How to Invest in Parag Parikh Liquid Fund Direct Growth?

Scheme: parag-parikh-elss-tax-saver-fund-direct-growth
  FAQ count: 8
   First question: How to Invest in Parag Parikh ELSS Tax Saver Fund Direct Growth?

Scheme: parag-parikh-conservative-hybrid-fund-direct-growth
  FAQ count: 8
   First question: How to Invest in Parag Parikh Conservative Hybrid Fund Direct Growth?

Scheme: parag-parikh-arbitrage-fund-direct-growth
  FAQ count: 8
   First question: How to Invest in Parag Parikh Arbitrage Fund Direct Growth?

Scheme: parag-parikh-dynamic-asset-allocation-fund-direct-growth
  FAQ count: 8
   First question: How to Invest in Parag Parikh Dynamic Asset Allocation Fund Direct Growth?


In [45]:
# === Cell 4: Save updated structured DB with FAQs ===

with STRUCTURED_DB_PATH.open("w", encoding="utf-8") as f:
    json.dump(db, f, ensure_ascii=False, indent=2)

print(f"\n✅ Updated structured DB saved to:\n{STRUCTURED_DB_PATH}")


✅ Updated structured DB saved to:
E:\ML Projects\RAG_Chatbot_PP_MF\project_files\data\json_files\parag_parikh_structured_db.json


In [46]:
# === Cell 5: Print FAQs for a specific scheme ===

TARGET_SCHEME_ID = "parag-parikh-arbitrage-fund-direct-growth"  # or whichever scheme you want

for row in db:
    if row["scheme_id"] == TARGET_SCHEME_ID:
        print(f"Scheme: {row['scheme_id']}\n")
        print("Overview scheme_name:", row.get("overview", {}).get("scheme_name"))
        print("\nFAQs:")
        pprint.pp(row.get("faqs"), width=120)
        break
else:
    print(f"Scheme not found: {TARGET_SCHEME_ID}")


Scheme: parag-parikh-arbitrage-fund-direct-growth

Overview scheme_name: Parag Parikh Arbitrage Fund Direct Growth

FAQs:
{'items': [{'question': 'How to Invest in Parag Parikh Arbitrage Fund Direct Growth?',
            'answer': 'You can easily invest in Parag Parikh Arbitrage Fund Direct Growth in a hassle-free manner on '
                      'Groww. The process is extremely simple, quick and completely paperless. Invest in a few minutes '
                      'with the following steps: \n'
                      ' * Log on to your Groww account\n'
                      ' * Search for Parag Parikh Arbitrage Fund Direct Growth from the search box \n'
                      ' * In order to invest, you will have to complete all the KYC formalities which are completely '
                      'online and paperless and take a few minutes to complete\n'
                      ' * Once you are done with that, you can start investing in Parag Parikh Arbitrage Fund Direct '
                 